# Ensemble Surfaces

No single backend is universally correct.  Road-network Dijkstra captures
turn costs and one-way streets; friction-raster MCP captures off-road
walking and handles rural areas.  An **ensemble** combines multiple
backends into a single surface, reducing backend-specific bias and
quantifying uncertainty.

**What's covered**
1. Mean, median, min, max ensembles
2. Weighted ensembles (prior knowledge about backend accuracy)
3. Per-pixel uncertainty — inter-model standard deviation
4. `compute_all_surfaces()` — inspect each backend individually
5. `nan_policy` — how to handle backends with partial coverage
6. Visualising ensemble surfaces and uncertainty maps


## Setup

In [ ]:
from idd_travel_time.access import (
    BoundingBox, OsmnxSurfaceRouter, FrictionSurfaceRouter,
    EnsembleSurfaceRouter,
)
from idd_travel_time.constants import TransportMode
import numpy as np, matplotlib.pyplot as plt

DESTINATION = (-1.3006, 36.8073)
BBOX = BoundingBox(west=36.76, south=-1.36, east=36.86, north=-1.24)
RESOLUTION = 0.01
MODE = TransportMode.WALKING

osmnx_router   = OsmnxSurfaceRouter()
friction_5kph  = FrictionSurfaceRouter(speed_kph_override=5.0)
friction_4kph  = FrictionSurfaceRouter(speed_kph_override=4.0)  # slower walking


## 1 · Compute individual surfaces

In [ ]:
print("Computing surfaces …")
s_osmnx  = osmnx_router.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)
s_f5     = friction_5kph.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)
s_f4     = friction_4kph.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)

for name, s in [("osmnx", s_osmnx), ("friction_5kph", s_f5), ("friction_4kph", s_f4)]:
    st = s.summary_stats()
    print(f"  {name:<20} mean={st['mean']:.1f} min  nan={st['nan_fraction']:.0%}")


## 2 · Mean ensemble

In [ ]:
ens_mean = EnsembleSurfaceRouter(
    [osmnx_router, friction_5kph, friction_4kph],
    strategy="mean",
)
s_mean = ens_mean.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)
print(f"Ensemble mean: {s_mean.summary_stats()['mean']:.1f} min")


## 3 · Weighted ensemble

In [ ]:
# Prior: trust osmnx most (urban Nairobi, good road coverage),
# friction_5kph second, friction_4kph least (too slow for this city)
ens_weighted = EnsembleSurfaceRouter(
    [osmnx_router, friction_5kph, friction_4kph],
    strategy="mean",
    weights=[0.5, 0.35, 0.15],
)
s_weighted = ens_weighted.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)
print(f"Weighted ensemble mean: {s_weighted.summary_stats()['mean']:.1f} min")


## 4 · Per-pixel uncertainty

In [ ]:
ens_unc = EnsembleSurfaceRouter(
    [osmnx_router, friction_5kph, friction_4kph],
    strategy="mean",
)
uncertainty = ens_unc.compute_uncertainty(DESTINATION, BBOX, RESOLUTION, MODE)
unc_stats = uncertainty.summary_stats()
print(f"Uncertainty surface: mean std dev = {unc_stats['mean']:.1f} min")
print(f"  Max uncertainty: {unc_stats['max']:.1f} min (pixels where backends disagree most)")


## 5 · Visualise all four

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
extent = [BBOX.west, BBOX.east, BBOX.south, BBOX.north]

panels = [
    (axes[0, 0], s_osmnx,    "OsmnxSurfaceRouter",         "RdYlGn_r", 0, 60),
    (axes[0, 1], s_mean,     "Ensemble (mean, equal wt.)",  "RdYlGn_r", 0, 60),
    (axes[1, 0], s_weighted, "Ensemble (weighted 0.5/0.35/0.15)", "RdYlGn_r", 0, 60),
    (axes[1, 1], uncertainty,"Uncertainty (inter-model std)", "hot_r",   0, 15),
]

for ax, surf, title, cmap, vmin, vmax in panels:
    im = ax.imshow(surf.data, cmap=cmap, vmin=vmin, vmax=vmax,
                   origin="upper", aspect="auto", extent=extent)
    ax.scatter([DESTINATION[1]], [DESTINATION[0]], c="blue", s=120, zorder=5)
    ax.set_title(title)
    label = "std dev (min)" if "Uncertainty" in title else "Travel time (min)"
    plt.colorbar(im, ax=ax, label=label)

plt.tight_layout()
plt.show()


## 6 · `compute_all_surfaces()` — per-backend dict

In [ ]:
ens = EnsembleSurfaceRouter([osmnx_router, friction_5kph], strategy="mean")
all_surfaces = ens.compute_all_surfaces(DESTINATION, BBOX, RESOLUTION, MODE)

print("Per-backend surfaces:")
for name, s in all_surfaces.items():
    print(f"  {name}: {s}")


## 7 · NaN policy — partial coverage backends

In [ ]:
# "propagate" (default): NaN in any backend → NaN output
ens_prop = EnsembleSurfaceRouter(
    [osmnx_router, friction_5kph],
    strategy="mean", nan_policy="propagate",
)
s_prop = ens_prop.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)

# "ignore": skip NaN backends per pixel
ens_ignore = EnsembleSurfaceRouter(
    [osmnx_router, friction_5kph],
    strategy="mean", nan_policy="ignore",
)
s_ignore = ens_ignore.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)

print(f"propagate nan_fraction: {s_prop.summary_stats()['nan_fraction']:.1%}")
print(f"ignore    nan_fraction: {s_ignore.summary_stats()['nan_fraction']:.1%}")
print()
print("Use 'ignore' when backends have complementary coverage gaps.")
